# Adicionar dados

## copiar caminhos no prompt

In [ ]:
# in_dir = str(input("Digite o caminho para o diretório de entrada: ")).replace("\\","/")
# out_dir = str(input("Digite o caminho para o diretório de saída: ")).replace("\\","/")
# projecao = str(input("Digite a projeção desejada. Caso queira usar o padrão digite: 0")).strip()
# if projecao == str(0):
#   projecao = None

## definir caminhos fixos

In [ ]:
#Digite o caminho para o diretório de entrada e sáida
in_dir = r"C:\Users\<user>\<input>" 
out_dir = r"C:\Users\<user>\<output>"
# caso queira especificar projecao
projecao = None # substituir aqui

# Gerenciar pacotes

## instalar pacotes via pip se necessário

In [ ]:
# %pip install <package name>
# %pip install fiona

## importar pacotes

In [ ]:
import geopandas as gpd
import fiona
import pandas as pd
from shapely.ops import polygonize
from glob import glob
import os

# Funções de trabalho

In [ ]:
def area_calc(gdf, crs=None, limiar=10):
  '''
  Calcula a área em hectares de cada feição em um GeoDataFrame e adiciona colunas
  para limites superior e inferior da área baseados em um limiar percentual.

  Parâmetros:
  ----------
  gdf : geopandas.GeoDataFrame
      O GeoDataFrame de entrada contendo as geometrias para cálculo da área.
  crs : str, opcional
      O sistema de referência de coordenadas (CRS) para projetar o GeoDataFrame
      antes do cálculo da área. Se `None`, usa 'Albers IBGE SIRGAS 2000' por padrão.
      Deve ser um CRS projetado para cálculos de área precisos.
  limiar : int ou float, opcional
      O limiar percentual (0-100) usado para calcular as áreas máximas e mínimas.
      Por padrão é 10 (10%).

  Retorna:
  -------
  geopandas.GeoDataFrame
      Um GeoDataFrame com as colunas 'area_ha', 'area_max' e 'area_min' adicionadas.
  '''
  # Verifica se um CRS específico foi fornecido
  if crs is None:
    # Define o WKT para a projeção 'Albers IBGE SIRGAS 2000' se nenhum CRS for especificado
    wkt = 'PROJCS["Albers_IBGE_SIRGAS2000",\
    GEOGCS["GCS_SIRGAS_2000",\
        DATUM["D_SIRGAS_2000",\
            SPHEROID["GRS_1980",6378137.0,298.257222101]],\
        PRIMEM["Greenwich",0.0],\
        UNIT["Degree",0.0174532925199433]],\
    PROJECTION["Albers_Conic_Equal_Area"],\
    PARAMETER["False_Easting",5000000.0],\
    PARAMETER["False_Northing",10000000.0],\
    PARAMETER["central_meridian",-54.0],\
    PARAMETER["standard_parallel_1",-12.5],\
    PARAMETER["standard_parallel_2",-22.5],\
    PARAMETER["latitude_of_origin",-32.0],\
    UNIT["Meter",1.0]]'

    # Projeta o GeoDataFrame para o CRS definido
    gdf_crs = gdf.to_crs(wkt)
    print("Projeção utilizada: Albers IBGE SIRGAS 2000")

  else:
    # Projeta o GeoDataFrame para o CRS fornecido
    gdf_crs = gdf.to_crs(crs)
    # Tenta verificar se o CRS é projetado; se falhar, assume que não é planar
    try:
      gdf_crs.crs.is_projected
    except:
      print(f"A projeção {gdf_crs.crs} não é planar (é geográfica/angular).")
    else:
      print(f"Projeção utilizada: {gdf_crs.crs}")

  # Calcula a área de cada feição em metros quadrados e converte para hectares
  gdf_crs["area_ha"] = gdf_crs.area / 10000
  # Converte o limiar percentual para um fator decimal
  perc =  (limiar/100)
  # Calcula a área máxima permitida com base no limiar
  gdf_crs["area_max"] = gdf_crs['area_ha'] * (1 + perc)
  # Calcula a área mínima permitida com base no limiar
  gdf_crs["area_min"] = gdf_crs['area_ha'] * (1 - perc)

  print(f"Área calculada em hectares e limiares máximos e mínimos aplicados com {limiar}% de variação.")

  return gdf_crs.to_crs('EPSG:3857')

In [ ]:
def unify_geometries(gdf):
  '''
  Unifica todas as geometrias de um GeoDataFrame em uma única geometria.
  A função mantém os atributos da primeira feição do GeoDataFrame original
  e substitui sua geometria pela união de todas as geometrias.

  Parâmetros:
  ----------
  gdf : geopandas.GeoDataFrame
      O GeoDataFrame de entrada com as geometrias a serem unificadas.

  Retorna:
  -------
  geopandas.GeoDataFrame
      Um novo GeoDataFrame contendo uma única linha com a geometria unificada
      e os atributos da primeira linha do GeoDataFrame original.
  '''
  # Cria uma cópia da primeira linha do GeoDataFrame original para manter os atributos
  gdf_union = gdf[0:1].copy()
  # 1. Corrige geometrias inválidas PRIMEIRO (com parênteses)
  geometrias_validas = gdf.geometry.make_valid()
  # 2. Une todas as geometrias corrigidas
  geometria_unica = geometrias_validas.union_all()
  # 3. Atribui o resultado encapsulado em uma lista para evitar erros de índice do pandas
  gdf_union['geometry'] = [geometria_unica]
  return gdf_union

In [ ]:
def poligonize_lines(gdf):
    """
    Converte geometrias lineares (LineString, MultiLineString) de um GeoDataFrame em polígonos.

    Esta função separa as geometrias lineares das demais, utiliza a operação de 
    poligonização para formar polígonos fechados a partir das linhas fornecidas, e 
    retorna um novo GeoDataFrame contendo os polígonos recém-criados combinados com 
    quaisquer geometrias não-lineares (como polígonos originais) que já existiam no 
    GeoDataFrame de entrada.

    Parâmetros:
    -----------
    gdf : geopandas.GeoDataFrame
        O GeoDataFrame de entrada contendo as geometrias a serem processadas.

    Retorno:
    --------
    geopandas.GeoDataFrame
        Um novo GeoDataFrame contendo os polígonos gerados e as geometrias 
        não-lineares originais. Retorna o próprio `gdf` original caso não existam 
        linhas ou não seja possível formar polígonos fechados.
    """
    
    # 1. Cria uma máscara booleana para identificar o que é linha
    is_line = gdf['geometry'].type.isin(["LineString", "MultiLineString"])
    is_plo = gdf['geometry'].type.isin(["Polygon", "MultiPolygon"])
    
    # 2. Divide o GeoDataFrame original em dois subconjuntos
    gdf_lines = gdf[is_line]         # Contém apenas as linhas
    gdf_pol = gdf[~is_line].copy()   # Contém o resto (polígonos, pontos, etc.)
    
    
    # 3. Validação inicial: verifica se há linhas para processar
    if gdf_lines.empty:
        print("Nenhuma geometria linear encontrada no GeoDataFrame.")
        return gdf
        
    print(f'Transformando {len(gdf_lines)} linhas em polígonos...')
    
    # 4. Processo de Poligonização
    # Extrai as geometrias das linhas para uma lista simples (requerido pelo Shapely)
    lines_list = gdf_lines['geometry'].tolist()
    # A função polygonize do Shapely encontra os anéis fechados formados pelas linhas
    polygons = list(polygonize(lines_list))
    
    # 5. Validação pós-processamento: verifica se alguma linha formou um polígono fechado
    if not polygons:
        print("Não foi possível formar nenhum polígono fechado a partir das linhas.")
        return gdf_pol
        
    # 6. Construção do novo GeoDataFrame com os resultados
    # Utiliza o mesmo sistema de coordenadas (CRS) do dataframe original
    gdf_poligonos = gpd.GeoDataFrame(geometry=polygons, crs=gdf.crs)
    # Cria um identificador único para os novos polígonos baseado no índice
    gdf_poligonos['id'] = gdf_poligonos.index
    
    print(f'Total de {len(gdf_poligonos)} polígonos criados com sucesso.')

    # 7. Mesclagem (Combinação) dos dados
    if not gdf_pol.empty:
        # Se existiam outras geometrias (ex: polígonos antigos), empilha os novos com os velhos
        # pd.concat é usado aqui por ser muito mais rápido e seguro que um 'merge' espacial
        gdf_result = pd.concat([gdf_pol, gdf_poligonos], ignore_index=True)
        return gdf_result
    else:
        # Se o dataframe original só tinha linhas, retorna apenas os novos polígonos
        return gdf_poligonos

In [ ]:
def ler_todas_camadas_kml(caminho_kml):
    """
    Lê todas as camadas (layers) de um arquivo KML e as combina em um único GeoDataFrame.

    Como o GeoPandas por padrão lê apenas a primeira camada de um arquivo KML, esta função 
    utiliza a biblioteca Fiona para listar todas as camadas disponíveis, itera sobre elas 
    fazendo a leitura individual e, por fim, concatena os resultados válidos.

    Parâmetros:
    -----------
    caminho_kml : str
        O caminho (path) para o arquivo .kml que será lido.

    Retorno:
    --------
    geopandas.GeoDataFrame ou None
        Um GeoDataFrame contendo todas as geometrias do KML empilhadas, com uma coluna 
        adicional ('nome_camada_origem') indicando de qual aba/pasta cada geometria veio.
        Retorna `None` se houver erro na leitura ou se todas as camadas estiverem vazias.
    """
    
    # 1. Configuração do Driver: Habilita o suporte para KML e LIBKML no Fiona.
    # Isso é obrigatório pois muitas instalações do fiona/gdal desativam o KML por padrão.
    fiona.drvsupport.supported_drivers['KML'] = 'rw'
    fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'

    try:
        # 2. Descoberta de Camadas: Lê os metadados do KML para listar os nomes das camadas.
        camadas = fiona.listlayers(caminho_kml)
        print(f"Total de {len(camadas)} camadas encontradas: {camadas}")
    except ValueError as e:
        # Tratamento de erro caso o arquivo não exista ou esteja corrompido.
        print(f"Erro ao ler o arquivo. Verifique se o caminho está correto: {e}")
        return None

    # Lista que servirá de contêiner para armazenar temporariamente os dados de cada camada.
    lista_gdfs = []

    # 3. Processamento Individual: Itera sobre cada nome de camada encontrado.
    for camada in camadas:
        print(f"Lendo camada: {camada}...")
        
        # Lê estritamente a camada atual da iteração.
        gdf_camada = gpd.read_file(caminho_kml, driver='KML', layer=camada)
        
        # 4. Filtragem de Dados: Verifica se a camada retornou algum dado válido.
        if not gdf_camada.empty:
            # Cria uma coluna de rastreabilidade para não perdermos a origem da geometria 
            # após juntarmos tudo em um arquivo só.
            gdf_camada['nome_camada_origem'] = camada 
            
            # Adiciona os dados válidos à nossa lista contêiner.
            lista_gdfs.append(gdf_camada)
        else:
            # É comum o Google Earth criar pastas/camadas apenas para organização visual (sem geometrias).
            print(f"  -> A camada '{camada}' está vazia e foi ignorada.")

    # 5. Consolidação: Junta todos os pedaços em um objeto final.
    if lista_gdfs:
        # O pd.concat é a forma mais rápida de empilhar DataFrames verticalmente.
        # ignore_index=True garante que o índice do novo DataFrame fique contínuo (0, 1, 2...).
        gdf_completo = pd.concat(lista_gdfs, ignore_index=True)
        print(f"\nSucesso! GeoDataFrame final criado com {len(gdf_completo)} registros.")
        return gdf_completo
    else:
        # Retorno de segurança caso o KML fosse composto inteiramente por pastas vazias.
        print("Nenhuma geometria válida foi encontrada em nenhuma das camadas.")
        return None

In [ ]:
def area_files_list(in_dir=str,out_dir=str,unify=True,crs=None,limiar=10):
  '''
  Processa todos os arquivos shapefile e KML/KMZ em um diretório de entrada,
  calcula suas áreas usando a função `area_calc`, e salva os resultados
  como novos shapefiles em um diretório de saída.

  Parâmetros:
  ----------
  in_dir : str
      O caminho para o diretório contendo os arquivos de entrada (shapefiles, KML/KMZ).
  out_dir : str
      O caminho para o diretório onde os arquivos de saída (shapefiles com áreas calculadas)
      serão salvos.
  unify: bool, opcional
      Se verdadeiro, as geometrias são unificadas, mantendo os atributos da primeira feição
      antes de calcular as áreas.
      Por padrão é True.
  crs : str, opcional
      O sistema de referência de coordenadas (CRS) a ser usado para o cálculo de área.
      Passado diretamente para `area_calc`. Se `None`, 'Albers IBGE SIRGAS 2000' é usado.
  limiar : int ou float, opcional
      O limiar percentual (entre 0-100) para calcular as áreas máximas e mínimas, passado para `area_calc`.
      Por padrão é 10 (10%).
  '''

  # Encontra todos os arquivos shapefile (*.shp) no diretório de entrada
  list_shapefiles = glob(os.path.join(in_dir,"*.shp"))
  print(f"{len(list_shapefiles)} shapefiles encontrados.\n")

  # Encontra todos os arquivos KML/KMZ (*.km* - .kml, .kmz) no diretório de entrada
  list_kml = glob(os.path.join(in_dir,"*.km*"))
  print(f"{len(list_kml)} kml ou kmz encontrados.\n")

  # Combina as listas de shapefiles e KML/KMZ para processamento
  list_files = list_shapefiles + list_kml

  # Garante que o diretório de saída exista; se não, ele é criado
  os.makedirs(out_dir, exist_ok=True)

  # Itera sobre cada arquivo na lista para processamento
  for file in list_files:
    # Extrai o nome do arquivo e o nome base (sem extensão)
    name = os.path.split(file)[-1]
    out_name = name.split(".")[0]
    type_name = name.split(".")[-1]
    print(f"Processando {name}")

    # Lê o arquivo geoespacial em um GeoDataFrame
    if type_name != '.shp':
      gdf = ler_todas_camadas_kml(file)
    else:
        gdf = gpd.read_file(file)

    # aplica conversão de linhas para polígonos
    gdf = poligonize_lines(gdf)

    #unificar geometrias
    if unify:
        gdf = unify_geometries(gdf)
        print("Poligonos unificados em uma único multipolígono")
    
    # Chama a função area_calc para calcular as áreas e limiares
    gdf_area = area_calc(gdf=gdf,crs=crs,limiar=limiar)
    gdf_area = gdf_area[["area_ha","area_max","area_min","geometry"]]

    # Salva o GeoDataFrame resultante com as áreas em um novo shapefile no diretório de saída
    print(gdf_area.columns)
    gdf_area.to_file(os.path.join(out_dir,f"{out_name}.shp"))
    print(f"Arquivo {out_name}.shp criado em {out_dir}.\n")

  print("Processamento concluído")

# Aplicação das funções

In [ ]:
area_files_list(in_dir=in_dir, out_dir=out_dir, crs=projecao)

#